## STOCHASTIC MODELING
MODULE 6 | LESSON 1


---



# **Markov Decision Processes (MDP) and Reinforcement Learning basics**

|  |  |
|:---|:---|
|**Reading Time** |  70 minutes |
|**Prior Knowledge** | HMM, Markov Property, Transition probabilities |
|**Keywords** | MDP, Policy iteration


---

In the previous module we dealt with HMM, using the Hamilton filter and EM algorithm to identify specific market regimes. This is what is usually known as "an inference problem". However, there's another important decision following the inference, that is the "control problem". In plain words, how should we proceed given our inferences about the market regime? This is where Markov Decision Processes (MDP) are key.

In [1]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1129185625", h="3298dbabb7", width=700, height=450) 

In [2]:
# Import libraries to be used later on
import numpy as np

## **1. Markov Decision Processes (MDP)**

A Markov Decision Process (MDP, henceforth) is a mathematical framework for modelling the sequential decision-making under uncertainty. This is, arguably, pretty similar to the decision problem of an investor.

Formally, a MDP is defined by the tuple $(S, A, P, R, \gamma)$:

- $S$: **State space** (finite or infinite set of possible states)

- $A$: **Action space** (set of available actions)

- $P$: **Transition function**, where $P(s' | s, a)$ gives the probability of transitioning to state $s'$ from state $s$ when taking action $a$.

- $R$: **Reward function**, $R(s, a)$ specifying immediate reward from transtion to state $s$ after action $a$.

- $\gamma \in [0, 1]$: **Discount factor** for future rewards.


As you can expect, just as in HMMs, the Markov property holds in MDPs: the future depends only on the current state, not the history. However, now our actions influence the transition probabilities:

\
$$
P(s_{t+1} | s_t , a_t, s_{t-1}, a_{t-1}, ..., s_0, a_0) = P(s_{t+1} | s_t, a_t)
$$

\
So, how do we tell the investor what actions should she implement in each state? That is the role of the **policy**.


### 1.1. Policies

A **policy** $\pi$ defines the investor's behavior at each state by mapping states to actions. The polciy of a MDP can be:

- **Deterministic**: $\pi : S → A$ (one defined action per state)

- **Stochastic**: $\pi(a | s)$ gives the probability of taking action $a$ in state $s$.

In our context you can think of a policy as a trading rule. For instance, in the previous module we developed a HMM and tested its power with a trading strategy where we ex-ante set a rule that we would *"buy or sell when the probability of a certain future scenario was above or below a threshold"*.

There are infinite policies, of course, and you have probably heard of some in forms such as *"reduce exposure when volatility spikes"*, or *"buy when RSI is below 30"*. These policies of course depend on the possible set of actions that the investor can take. When using MDPs, our objective is most often to find the optimal set of policies that maximize returns from any starting state. In order to find those, a first step is unveiling **value functions**

### 1.2. Value functions

Value functions give us a sense of the value of expected cumulative rewards. We must differenciate between 2 different value functions, each of them giving us distinct (although related) information:


1. **State-value function**, $V^\pi (s)$ (often referred to as V-function).

State-value function under policy $\pi$ measures the expected cumulative reward from state $s$:

$$
V^\pi (s) = \mathbb{E}_\pi \left[ \sum_{t=0}^\infty \gamma^t r_t | s_0=s \right]
$$

\
The state-value function essentially answers the question of **how good is to be in state $s$ under policy $\pi$?**

One important limitation of V-function is that it does not tell us anything regarding which specific action to take, or which action adds more value. For that, we need the **action-value function**.

\
2. **Action-value function**, $Q^\pi (s|a)$ (often referred to as Q-function).

Action-value function incorporates actions into the value function framework:

$$
Q^\pi (s|a) = \mathbb{E}_\pi \left[ \sum_{t=0}^\infty \gamma^t r_t | s_0=s, a_0 = a \right]
$$

\
This way, Q-function incorporates the inmediate action choice into the overall value. Specifically, Q-function answers the question of **how good is to take action $a$ in state $s$ and then follow policy $\pi$?**

As you can already guess, there is a fundamental relationship between V- and Q-functions, such as:

$$
V^\pi = \sum_a \pi(a |s) ⋅ Q^\pi (s,a)
$$

This is relatively straighforward for deterministic policies:

$$
V^\pi (s) = Q^\pi(s, \pi(s))
$$

Conversely, we can also express Q-function via V-function:

$$
Q^\pi(s,a) = \sum_{s'} P(s' |s,a) \left[ R(s,a) + \gamma V^\pi(s') \right]
$$

\
As we will see shortly in detail, each of these functions has an specific use.

- We will use V-function for evaluating the merits of a given policy. For a portfolio manager this could be answering a question such as *"how good is our portfolio's current allocation?"*.

- In turn, we will focus on Q-function when we want our model to learn about policies that maximize rewards. In the spirit of the previous example for a portfolio manager, Q-function will help us answer questions such as *"should we buy or sell gien the current market conditions?"*

## **2. A simple investment MDP example**

To illustrate all these concepts in practice, let's work on a very basic example of a MDP for an investor. We will have time to complicate things in the upcoming lessons. For now, let's think of an investor that is facing 3 potential market regimes: **Bull**, **Bear**, and **Crash**. At each point in time, the investor can choose to **buy** or **sell** the overall market, or **hold** positions in cash. With this information we can set up the states and actions of the MDP:

In [3]:
# State Space - Market Regimes
states = ['Bull', 'Bear', 'Crash']

# Action Space - Trading Decisions
actions = ['Buy', 'Hold', 'Sell']

We should then gather transition probabilities, which now depend not only on the previous state, but also on the actions taken by the investor. A possible set of probabilities, together with brief rationales, could be the following:

In [4]:
# Transition probabilities P(s'|s,a)

transitions = {
    # Bull market transitions
    ('Bull', 'Buy'): {
        'Bull': 0.7,   # Momentum trading reinforces bull trend
        'Bear': 0.2,   # Normal correction probability
        'Crash': 0.1   # Bubble risk from overheating
    },
    ('Bull', 'Hold'): {
        'Bull': 0.6,   # Natural trend continuation
        'Bear': 0.3,   # Higher correction chance without support
        'Crash': 0.1   # Baseline crash risk
    },
    ('Bull', 'Sell'): {
        'Bull': 0.5,   # Selling pressure reduces bull persistence
        'Bear': 0.4,   # Increased bear probability from selling
        'Crash': 0.1   # Crash risk unchanged
    },

    # Bear market transitions
    ('Bear', 'Buy'): {
        'Bull': 0.3,   # Buying can help/trigger recovery
        'Bear': 0.6,   # Bear trend still likely to persist
        'Crash': 0.1   # Reduced crash risk from support
    },
    ('Bear', 'Hold'): {
        'Bull': 0.2,   # Natural recovery probability
        'Bear': 0.7,   # Bears tend to persist without intervention
        'Crash': 0.1   # Baseline crash risk
    },
    ('Bear', 'Sell'): {
        'Bull': 0.1,   # Selling pressure prevents recovery
        'Bear': 0.6,   # Continued bear market
        'Crash': 0.3   # Panic selling can trigger crash!
    },

    # Crash transitions
    ('Crash', 'Buy'): {
        'Bull': 0.2,   # "Buy the dip" can lead to recovery
        'Bear': 0.7,   # Usually crashes transition to bear
        'Crash': 0.1   # Buying prevents extended crash
    },
    ('Crash', 'Hold'): {
        'Bull': 0.1,   # Rare V-shaped recovery
        'Bear': 0.7,   # Natural transition to bear market
        'Crash': 0.2   # Some crash persistence
    },
    ('Crash', 'Sell'): {
        'Bull': 0.0,   # Selling prevents any recovery
        'Bear': 0.5,   # Might stabilize into bear
        'Crash': 0.5   # Selling in crash extends crash!
    }
}

And, finally, the rewards for each state-action pair. Of course, we should estimate this from data, but let's for now assume these would be them (in % points):

| State | Buy  | Hold | Sell |
|-------|------|------|------|
| Bull  | +2.0 | +0.5 | -0.5 |
| Bear  | -1.0 | 0.0  | +1.0 |
| Crash | -5.0 | -0.5 | +3.0 |

In [5]:
# Rewards R(s,a) - in percentage points!

rewards = {
    # Bull market rewards
    ('Bull', 'Buy'):   +2.0,   # Long position in rising market (best case)
    ('Bull', 'Hold'):  +0.5,   # Risk-free rate / minimal participation
    ('Bull', 'Sell'):  -0.5,   # Short position loses in bull market

    # Bear market rewards
    ('Bear', 'Buy'):   -1.0,   # Long position loses in falling market
    ('Bear', 'Hold'):   0.0,   # Neutral, avoiding losses
    ('Bear', 'Sell'):  +1.0,   # Short position profits from decline

    # Crash rewards
    ('Crash', 'Buy'):  -5.0,   # Severe losses trying to catch falling knife
    ('Crash', 'Hold'): -0.5,   # Even "safe" assets lose in crash
    ('Crash', 'Sell'): +3.0    # Large profit from short position/puts
}

In this simple setting we can now play a bit with the calculation of V- and Q-functions. Let's assume for now that $\gamma=0.95$ and a simple deterministic policy such that $\pi = \left(Bull:buy, Bear:sell, Crash:hold \right)$

- Could you calculate, $V^\pi (Bear)$, that is, the value function from this policy for state *Bear*?

**Trick question!** Recall that $V^\pi(s)$ represents the expected cumulative discounted rewards from state $s$. Thus, calculating $V^\pi(Bear)$ requires knowing all future rewards along potentially infinite trajectories through *Bull*. *Bear*, and *Crash* states, which poses a computational problem. Here's the catch: to compute $V^\pi(Bear)$ we need to also know $V^\pi(Bull)$ and $V^\pi(Crash)$, which also depend on $V^\pi(Bear)$.

This circular dependency makes direct computation impossible—we can't trace infinite paths, and we can't solve for one state's value without knowing the others. We need a systematic approach that captures the recursive nature of value functions: the insight that today's value equals immediate reward plus discounted future value. This is precisely what **Bellman equations** provide—they transform our infinite-horizon problem into a solvable system of equations by exploiting this recursive structure.

## **3. Bellman equations**

The key insight of Bellman is recognizing that the value functions **must satisfy** a simple recursive relationship: *the value of being in a state equals the inmediate reward plus the discounted value of where you end up*. This is:

$$
V^\pi (s) = R(s, \pi(s)) + \gamma \sum_{s'} P(s' | s, \pi(s)) \cdot V^\pi (s')
$$

\
This idea also applies to Q-function of action-values:

$$
Q^\pi (s, a) = R(s,a) + \gamma \sum_{s'} P(s' | s, a) \cdot V^\pi(s')
$$

\
In our previous example, the Bellman equation for $V^\pi (Bear)$, where $\pi$ says 'sell', would be:

$$
V^\pi (Bear) = 1.0 + 0.95 \cdot [0.1 V^\pi(Bull) + 0.6 V^\pi(Bear) + 0.3 V^\pi(Crash)]
$$

\
And, if we wanted to know the value of subsequently taking action 'buy', the Q-function would be:

$$
Q^\pi (Bear, Buy) = -1.0 + 0.95 \cdot [0.3 V^\pi(Bull) + 0.6 V^\pi(Bear) + 0.1 V^\pi(Crash)]
$$

\
Bellman's approach give us a set of equations so that we can solve for value functions, allowing us to evaluate the value of a given policy and, more importantly, solving for the optimal policy.

Intuitively, the **optimal policy** depends on the optimal value function, $V^*$, which satisfies:

$$
V^*(s) = \max_a \left[R(s,a) + \gamma \sum_{s'} P(s' | s,a)\cdot V^*(s') \right]
$$

\
In simple words, the optimal value is achieved by taking the best action at each state. We can solve this with iteration methods, to find the optimal policy, $\pi^*$:

$$
\pi^* (s) = arg \max_a \left[R(s,a) + \gamma \sum_{s'} P(s' | s,a)\cdot V^*(s') \right]
$$

## 🔧 Code Task: Computing Q-values from V-function

Your task is to implement a function that computes the Q-value (action-value) for a specific state-action pair, given a V-function. This implements the Bellman equation relationship between Q and V functions.

Recall the Bellman equation for Q-function:

$$Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \cdot V^\pi(s')$$

**Function Requirements:**
- **Input:**
  - `state` (str): The current state (e.g., 'Bull', 'Bear', or 'Crash')
  - `action` (str): The action to evaluate (e.g., 'Buy', 'Hold', or 'Sell')
  - `V` (numpy array): The V-function values for all states, shape (3,)
  - `gamma` (float): Discount factor (default = 0.95)
  
- **Output:**
  - `q_value` (float): The computed Q-value for the given state-action pair

**Context:** 
You have access to the following pre-defined variables from the lesson:
- `states = ['Bull', 'Bear', 'Crash']`
- `actions = ['Buy', 'Hold', 'Sell']`
- `transitions`: dictionary with transition probabilities P(s'|s,a)
- `rewards`: dictionary with immediate rewards R(s,a)
- `s2i`: dictionary mapping state names to indices

In [6]:
# Starter Code

import numpy as np

def compute_q_value(state, action, V, gamma=0.95):
    """
    Compute Q-value for a given state-action pair using the Bellman equation.
    
    Parameters:
    - state (str): Current state name
    - action (str): Action to evaluate
    - V (numpy array): Value function for all states, shape (3,)
    - gamma (float): Discount factor
    
    Returns:
    - q_value (float): The computed Q-value
    """
    # TODO: Implement your logic here
    pass

## **4. Solving for the optimal policy: Policy iteration**

In order to solve for the optimal policy at each state, we will rely on policy iteration. This means going through the following process:

1. **Initialize** with any policy $\pi_0$.
2. **Policy evaluation**: find $V^{\pi_0}$ using Bellman
3. **Policy improvement**: check if we can improve the policy by computing Q-values for all possible actions in each state.
4. Evaluate the new  $\pi_1$
5. Check for futher improvement

(...)

This loop ends when there is **convergence**, in other words, when there are no improvements possible. Let's illustrate this with the previous example. Then we can code it in to get the optimal policy.


1. Set initial policy

We will set an initial conservative policy of 'buy and hold' that only buys in bull markets and otherwise holds cash:

$$
\pi_0 = [Bull: buy, Bear: hold, Crash: hold]
$$


2. Policy evaluation: check values of $\pi_0$ using Bellman for each state:

$$
V^{\pi_0} (Bull) = 2.0 + 0.95 \cdot [ 0.7 \cdot V^{\pi_0} (Bull) + 0.2 \cdot V^{\pi_0} (Bear) + 0.1 \cdot V^{\pi_0} (Crash)]
$$

$$
V^{\pi_0} (Bear) = 0.0 + 0.95 \cdot [ 0.2 \cdot V^{\pi_0} (Bull) + 0.7 \cdot V^{\pi_0} (Bear) + 0.1 \cdot V^{\pi_0} (Crash)]
$$

$$
V^{\pi_0} (Crash) = -0.5 + 0.95 \cdot [ 0.1 \cdot V^{\pi_0} (Bull) + 0.7 \cdot V^{\pi_0} (Bear) + 0.2 \cdot V^{\pi_0} (Crash)]
$$

\
Solving this system of equations will give us $V^{\pi_0} (Bull)\approx 16.47$ , $V^{\pi_0} (Bear) \approx 12.67$, and $V^{\pi_0} (Crash) \approx 11.71$


3. Policy improvement, let's calculate Q-values for each possible action in each state using $V^{\pi_0}(s)$:


- For Bull state:
$$
Q^{\pi_0}(Bull, buy) = 2.0 + 0.95 \cdot [0.7 \cdot 16.47 + 0.2 \cdot 12.67 + 0.1 \cdot 11.71] = 16.47
$$

$$
Q^{\pi_0}(Bull, hold) = 0.5 + 0.95 \cdot [0.6 \cdot 16.47 + 0.3 \cdot 12.67 + 0.1 \cdot 11.71] = 14.61
$$

$$
Q^{\pi_0}(Bull, sell) = -0.5 + 0.95 \cdot [0.5 \cdot 16.47 + 0.4 \cdot 12.67 + 0.1 \cdot 11.71] = 13.25
$$

The highest value when Bull market is in action **buy**, which aligns with our policy.

- We then repeat the process for Bear state:

$$
Q^{\pi_0}(Bear, buy) = 12.02
$$
$$
Q^{\pi_0}(Bear, hold) = 12.67
$$
$$
Q^{\pi_0}(Bear, sell) = 13.12
$$

\
In this case, the highest value is in action **sell**, which does not align with our initial policy. Hence, we can improve the current policy.

As you can guess, we will next calculate things for the Crash state to see if any improvements are possible in there. Next, we will reshape our policy (e.g. by deciding to sell when in Bear or Crash states) and re-evaluate. Then, we will look for potential improvements again. This iterative process ends when there are no improvements possible. That final policy will be the optimal one, $\pi^*$.

Of course, this is a tedious process to do by hand. But it's relatively straightforward to do in coding. Let's see that next, solving our investor decision problem.

## **5. Solving the investment MDP example with code**

Let's start by hardcoding the information needed. Note that some of these variables have already been defined before (e.g., transition probs, actions, states, or rewards).

In [7]:
# Set gamma and define dictionaries that "translate" states and actions to numbers for simplicity in coding
gamma = 0.95
s2i = {s:i for i,s in enumerate(states)}
a2i = {a:i for i,a in enumerate(actions)}

In [8]:
# Pack transition probs and rewards into arrays for more efficient coding later on

A = len(actions); S = len(states)
P = np.zeros((A, S, S))
r = np.zeros((A, S))

for a in actions:
    ai = a2i[a]
    for s in states:
        si = s2i[s]
        # Fill transition row
        row = np.zeros(S)
        for sp, prob in transitions[(s,a)].items():
            row[s2i[sp]] = prob
        P[ai, si] = row
        # Reward for taking a in s
        r[ai, si] = rewards[(s,a)]

Now that we have all our data set, let's start defining the functions that will help us throughout. First, **policy evaluation**:

In [9]:
def policy_eval(policy, gamma=gamma):
    """
    policy: list[str] of length S, e.g. ['Buy','Hold','Hold']
    returns: Vπ (S,)
    """
    # Build Pπ and rπ
    P_pi = np.zeros((S, S))
    r_pi = np.zeros(S)
    for s in range(S):
        a = policy[s]
        ai = a2i[a]
        P_pi[s] = P[ai, s]
        r_pi[s] = r[ai, s]

    # Solve (I - γ Pπ) V = rπ --> Efficient way to solve the system of Bellman equations
    A_mat = np.eye(S) - gamma * P_pi
    V_pi = np.linalg.solve(A_mat, r_pi)
    return V_pi

Next, the computation of Q-functions:

In [10]:
def Q_function(V, gamma=gamma):
    """
    returns: Qπ matrix shape (S, A) given Vπ
    """
    Q = np.zeros((S, A))
    for s in range(S):
        for a in range(A):
            Q[s, a] = r[a, s] + gamma * P[a, s] @ V
    return Q


And **policy improvement**:

In [11]:
def policy_improv(V, deterministic=True):
    """
    Greedy improvement: π'(s) = argmax_a Qπ(s,a)
    returns: new policy (list[str])
    """
    Q = Q_function(V)
    best_a_idx = np.argmax(Q, axis=1)
    new_policy = [actions[j] for j in best_a_idx]
    return new_policy, Q

Onmce we have set these, we can proceed with the iteration process:

In [12]:
policy = ['Buy', 'Hold', 'Hold'] # Initial policy: buy when Bull, hold otherwise

converge = False
iteration = 0

while not converge:
    iteration += 1
    V = policy_eval(policy) # Policy evaluation
    new_policy, Q = policy_improv(V) # Policy improvement

    print(f"Iter {iteration}")
    print("  Policy:", policy)
    print("  Vπ   :", np.round(V, 5).tolist())
    # Optional: peek at Q for education (Bull row = Q[0], etc.)
    print("  Qπ(Bull):", dict(zip(actions, np.round(Q[0], 5).tolist())))
    print("  Qπ(Bear):", dict(zip(actions, np.round(Q[1], 5).tolist())))
    print("  Qπ(Crash):", dict(zip(actions, np.round(Q[2], 5).tolist())))
    print()

    if new_policy == policy:
        converge = True
    else:
        policy = new_policy

print("Converged optimal policy:", policy)
print("Optimal V*:", np.round(V, 5).tolist())

Iter 1
  Policy: ['Buy', 'Hold', 'Hold']
  Vπ   : [16.47619, 12.66667, 11.71429]
  Qπ(Bull): {'Buy': 16.47619, 'Hold': 14.61429, 'Sell': 13.25238}
  Qπ(Bear): {'Buy': 12.02857, 'Hold': 12.66667, 'Sell': 13.12381}
  Qπ(Crash): {'Buy': 7.66667, 'Hold': 11.71429, 'Sell': 14.58095}

Iter 2
  Policy: ['Buy', 'Sell', 'Sell']
  Vπ   : [36.97956, 35.67752, 37.99394]
  Qπ(Bull): {'Buy': 36.97956, 'Hold': 35.35587, 'Sell': 34.23217}
  Qπ(Bear): {'Buy': 33.48478, 'Hold': 34.36109, 'Sell': 35.67752}
  Qπ(Crash): {'Buy': 29.36109, 'Hold': 33.95746, 'Sell': 37.99394}

Converged optimal policy: ['Buy', 'Sell', 'Sell']
Optimal V*: [36.97956, 35.67752, 37.99394]


As we can see, our policy iteration algorithm quickly converges to an optimal policy for our investor. This optimal policy entails a 'buy' action when a Bull market and a 'sell' action otherwise.

## **6. Conclusion**

Well done! In this lesson we have introduced Markov Decision Processes (MDP) as a central component of Reinforcement learning (RL) algorithms. MDPs build on the foundations we developed in the previous module, but go a bit further in "recommending" and optimal action to the agent.

So far we have dealt with a very simple MDP. Let's change that and consider more complex problems and processes in the next lesson!

---
Copyright 2025 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
